In [1]:
# =========================
# DATA LOADING
# =========================
# Import pandas library for data handling
import pandas as pd

# Load dataset from CSV file
# - engine='python' handles complex CSV formatting
# - encoding='latin-1' ensures compatibility with special characters
# - on_bad_lines='skip' ignores malformed rows
df = pd.read_csv(
    "./books.csv",
    engine='python',
    encoding='latin-1',
    on_bad_lines='skip'
)

# Display dataset shape (rows, columns)
print(df.shape)


(11119, 12)


In [2]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPrÃ©,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPrÃ©,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPrÃ©,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPrÃ©,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [3]:
# let's check the info of the dataset 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11119 entries, 0 to 11118
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11119 non-null  int64  
 1   title               11119 non-null  object 
 2   authors             11119 non-null  object 
 3   average_rating      11119 non-null  float64
 4   isbn                11119 non-null  object 
 5   isbn13              11119 non-null  int64  
 6   language_code       11119 non-null  object 
 7     num_pages         11119 non-null  int64  
 8   ratings_count       11119 non-null  int64  
 9   text_reviews_count  11119 non-null  int64  
 10  publication_date    11119 non-null  object 
 11  publisher           11119 non-null  object 
dtypes: float64(1), int64(5), object(6)
memory usage: 1.0+ MB


**Dataset Description**

- The dataset used in this project is a Goodreads books dataset that contains structured information about books. It includes 11,119 records and 12 columns, covering details such as title, authors, average rating, ratings count, language, and publication date. Each row represents a single book, and the dataset is clean, with no missing values across the columns. This makes it reliable and easy to use without heavy preprocessing.

For this system, the most useful attributes are:

- title → identifies the book
- authors → used for matching user preferences
- average_rating → used for ranking and similarity

Although the dataset does not include genre information, it still supports a meaningful recommendation system using author similarity and rating-based comparison. This fits well with the project goal of building a system that is simple, clear, and easy to explain.

The dataset is also small in size (around 1 MB), which allows fast execution in a Jupyter Notebook. This is helpful for demonstration, testing, and step-by-step explanation of how the system works.

What We Will Build Next

Now that the dataset is loaded, the next step is to build the core structure of the system.

We will implement:

- A Book class
- Represents each book with attributes like title, author, and rating
- A UserProfile class
- Stores user preferences such as favorite authors and rated books
- A Dataset transformation step
- Converts the pandas DataFrame into structured Book objects

This step is important because it turns raw data into usable system components. It also directly follows the class diagram we defined earlier. Once these components are ready, we will move to the recommendation logic, where the system will start generating suggestions based on user preferences.

### Exploratory Data Analysis (EDA)

- Before building system components, a small and focused EDA step helps ensure the dataset behaves as expected. The goal here is not deep statistical analysis, but to validate structure, understand distributions, and prepare clean inputs for the system.

In [4]:
# Shape and columns
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Shape: (11119, 12)

Columns: ['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


In [5]:
# =========================
# DATA EXPLORATION
# =========================
# Identify duplicate book titles
duplicates = df[df.duplicated(subset=['title'])]
print("Duplicate titles:", len(duplicates))

Duplicate titles: 775


In [6]:
# Display summary statistics for ratings
df['average_rating'].describe()

count    11119.000000
mean         3.934135
std          0.350384
min          0.000000
25%          3.770000
50%          3.960000
75%          4.135000
max          5.000000
Name: average_rating, dtype: float64

In [7]:
# Count number of unique authors
print("Unique authors:", df['authors'].nunique())

# Display top 10 most frequent authors
df['authors'].value_counts().head(10)



Unique authors: 6635


authors
P.G. Wodehouse      40
Stephen King        40
Rumiko Takahashi    39
Orson Scott Card    35
Agatha Christie     33
Piers Anthony       30
Sandra Brown        29
Mercedes Lackey     29
Dick Francis        28
Terry Pratchett     23
Name: count, dtype: int64

In [8]:
# Check for missing values in each column
df.isnull().sum()



bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
  num_pages           0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

In [9]:

# =========================
# DATA CLEANING
# =========================
# Select only relevant columns for the system
df_clean = df[['title', 'authors', 'average_rating']].copy()

# Rename columns for consistency with system design
df_clean = df_clean.rename(columns={
    'authors': 'author',
    'average_rating': 'rating'
})

# Remove rows with missing values
df_clean = df_clean.dropna()

# Display cleaned data sample
df_clean.head()



,title,author,rating
0,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPrÃ©,4.57
1,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPrÃ©,4.49
2,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42
3,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPrÃ©,4.56
4,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPrÃ©,4.78


In [10]:
# Remove duplicate titles to ensure uniqueness
df_clean = df_clean.drop_duplicates(subset=['title'])

# Display final dataset shape after cleaning
print("After cleaning:", df_clean.shape)

After cleaning: (10344, 3)


### Core System Entities

- Convert data → objects
- Follow your class diagram (Entity layer)
- Build a system, not just analysis

In [11]:
# =========================
# CORE SYSTEM ENTITIES
# =========================

# Book Class
# Represents a single book with basic attributes used for recommendation
class Book:
    def __init__(self, title, author, rating):
        # Initialize book attributes
        self.title = title
        self.author = author
        self.rating = rating

    def __repr__(self):
        # Return readable representation of the book
        return f"{self.title} by {self.author} (Rating: {self.rating})"



In [12]:

# UserProfile Class
# Stores user preferences including favorite authors and rated books
class UserProfile:
    def __init__(self):
        # List of preferred authors
        self.favorite_authors = []

        # Dictionary storing rated books {title: rating}
        self.rated_books = {}

    def add_favorite_author(self, author):
        # Add author to favorites if not already present
        if author not in self.favorite_authors:
            self.favorite_authors.append(author)

    def rate_book(self, title, rating):
        # Store or update user rating for a book
        self.rated_books[title] = rating

    def __repr__(self):
        # Return readable summary of user profile
        return f"Favorite Authors: {self.favorite_authors}\nRated Books: {self.rated_books}"



In [13]:

# =========================
# DATA TRANSFORMATION
# =========================
# Convert cleaned dataset into Book objects

books = []

for _, row in df_clean.iterrows():
    # Create Book object from each row
    book = Book(
        title=row['title'],
        author=row['author'],
        rating=row['rating']
    )
    books.append(book)

# Display total number of books loaded
print("Total books loaded:", len(books))

# Display sample books
books[:5]



Total books loaded: 10344


[Harry Potter and the Half-Blood Prince (Harry Potter  #6) by J.K. Rowling/Mary GrandPrÃ© (Rating: 4.57),
 Harry Potter and the Order of the Phoenix (Harry Potter  #5) by J.K. Rowling/Mary GrandPrÃ© (Rating: 4.49),
 Harry Potter and the Chamber of Secrets (Harry Potter  #2) by J.K. Rowling (Rating: 4.42),
 Harry Potter and the Prisoner of Azkaban (Harry Potter  #3) by J.K. Rowling/Mary GrandPrÃ© (Rating: 4.56),
 Harry Potter Boxed Set  Books 1-5 (Harry Potter  #1-5) by J.K. Rowling/Mary GrandPrÃ© (Rating: 4.78)]

In [14]:

# =========================
# USER PROFILE INITIALIZATION
# =========================
# Create sample user and define preferences

user = UserProfile()

# Add favorite authors
user.add_favorite_author("J.K. Rowling")
user.add_favorite_author("Stephen King")

# Add book ratings
user.rate_book("Harry Potter and the Half-Blood Prince", 5)
user.rate_book("The Shining", 4)

# Display user profile
print(user)

Favorite Authors: ['J.K. Rowling', 'Stephen King']
Rated Books: {'Harry Potter and the Half-Blood Prince': 5, 'The Shining': 4}


### Similarity Strategy Implementation

We will implement:

A base class: SimilarityStrategy
- Two concrete strategies:
- AuthorSimilarity
- RatingSimilarity

Each strategy:

- Computes its own score
- Can be independently modified or extended
- Contributes to the final recommendation score

In [15]:
# =========================
# SIMILARITY STRATEGY LAYER
# =========================

# Base Strategy Class
# Defines a common interface for all similarity calculation strategies
class SimilarityStrategy:
    def calculate(self, book, user_profile):
        # Abstract method to be implemented by subclasses
        raise NotImplementedError("Subclasses must implement this method")


# Author Similarity Strategy
# Checks if the book author matches user's favorite authors
class AuthorSimilarity(SimilarityStrategy):
    def calculate(self, book, user_profile):
        # Return strong match score if author is preferred
        if book.author in user_profile.favorite_authors:
            return 1.0
        return 0.0


# Rating Similarity Strategy
# Compares book rating with user's average rating preference
class RatingSimilarity(SimilarityStrategy):
    def calculate(self, book, user_profile):
        # If no ratings exist, similarity cannot be computed
        if not user_profile.rated_books:
            return 0.0

        # Compute average rating given by user
        avg_user_rating = sum(user_profile.rated_books.values()) / len(user_profile.rated_books)

        # Calculate difference between book rating and user preference
        difference = abs(book.rating - avg_user_rating)

        # Convert difference into similarity score (normalized between 0 and 1)
        return max(0, 1 - (difference / 5))



In [16]:

# =========================
# STRATEGY TESTING
# =========================
# Initialize strategy objects
author_strategy = AuthorSimilarity()
rating_strategy = RatingSimilarity()

# Select a sample book
sample_book = books[0]

# Display results of individual strategies
print("Book:", sample_book)
print("Author Score:", author_strategy.calculate(sample_book, user))
print("Rating Score:", rating_strategy.calculate(sample_book, user))

Book: Harry Potter and the Half-Blood Prince (Harry Potter  #6) by J.K. Rowling/Mary GrandPrÃ© (Rating: 4.57)
Author Score: 0.0
Rating Score: 0.986


Next, we Are Building
**RecommendationController**

Responsibilities:

- Loop through all books
- Apply similarity strategies
- Combine scores
- Exclude already rated books
- Rank results
- Return top recommendations

In [17]:
# =========================
# RECOMMENDATION CONTROLLER
# =========================

# Controller responsible for coordinating recommendation logic
class RecommendationController:
    def __init__(self, strategies, weights):
        # List of similarity strategies (e.g., author, rating)
        self.strategies = strategies

        # Corresponding weights for each strategy
        self.weights = weights

    def recommend(self, books, user_profile, top_n=5):
        # List to store recommendation results
        recommendations = []

        for book in books:
            # Skip books already rated by the user
            if book.title in user_profile.rated_books:
                continue

            total_score = 0
            explanation = {}

            # Apply each similarity strategy
            for strategy, weight in zip(self.strategies, self.weights):
                score = strategy.calculate(book, user_profile)

                # Weighted contribution to final score
                total_score += score * weight

                # Store individual strategy contribution for explanation
                explanation[strategy.__class__.__name__] = round(score, 3)

            # Append recommendation result
            recommendations.append({
                "book": book,
                "score": round(total_score, 3),
                "explanation": explanation
            })

        # Sort recommendations by score (highest first)
        recommendations.sort(key=lambda x: x["score"], reverse=True)

        # Return top N recommendations
        return recommendations[:top_n]



In [18]:

# =========================
# CONTROLLER INITIALIZATION
# =========================

# Initialize similarity strategies
author_strategy = AuthorSimilarity()
rating_strategy = RatingSimilarity()

# Combine strategies with weights
strategies = [author_strategy, rating_strategy]
weights = [0.6, 0.4]

# Create controller instance
controller = RecommendationController(strategies, weights)



In [19]:

# =========================
# GENERATE RECOMMENDATIONS
# =========================

# Run recommendation process
results = controller.recommend(books, user, top_n=5)

# Display raw results
for i, rec in enumerate(results, 1):
    print(f"\n{i}. {rec['book']}")
    print(f"   Score: {rec['score']}")
    print(f"   Explanation: {rec['explanation']}")




1. Harry Potter y la Orden del FÃ©nix (Harry Potter  #5) by J.K. Rowling (Rating: 4.49)
   Score: 0.999
   Explanation: {'AuthorSimilarity': 1.0, 'RatingSimilarity': 0.998}

2. Harry Potter Y La Piedra Filosofal (Harry Potter  #1) by J.K. Rowling (Rating: 4.47)
   Score: 0.998
   Explanation: {'AuthorSimilarity': 1.0, 'RatingSimilarity': 0.994}

3. Harry Potter and the Philosopher's Stone (Harry Potter  #1) by J.K. Rowling (Rating: 4.47)
   Score: 0.998
   Explanation: {'AuthorSimilarity': 1.0, 'RatingSimilarity': 0.994}

4. Carrie / 'Salem's Lot / The Shining by Stephen King (Rating: 4.54)
   Score: 0.997
   Explanation: {'AuthorSimilarity': 1.0, 'RatingSimilarity': 0.992}

5. The Green Mile by Stephen King (Rating: 4.44)
   Score: 0.995
   Explanation: {'AuthorSimilarity': 1.0, 'RatingSimilarity': 0.988}


In [20]:

# =========================
# OUTPUT FORMATTING (UI LAYER)
# =========================

# Convert explanation scores into user-friendly reasons
def format_explanation(explanation):
    reasons = []

    if explanation.get("AuthorSimilarity", 0) > 0:
        reasons.append("Matches your favorite author")

    if explanation.get("RatingSimilarity", 0) > 0.7:
        reasons.append("Rating is close to your preference")

    return reasons


# Display formatted recommendation output
for i, rec in enumerate(results, 1):
    book = rec['book']
    score = rec['score']
    explanation = rec['explanation']

    print(f"\n{i}. {book.title}")
    print(f"   Author: {book.author}")
    print(f"   Rating: {book.rating}")
    print(f"   Score: {score}")

    # Generate readable reasons
    reasons = format_explanation(explanation)

    print("   Reason:")
    for r in reasons:
        print(f"   - {r}")


1. Harry Potter y la Orden del FÃ©nix (Harry Potter  #5)
   Author: J.K. Rowling
   Rating: 4.49
   Score: 0.999
   Reason:
   - Matches your favorite author
   - Rating is close to your preference

2. Harry Potter Y La Piedra Filosofal (Harry Potter  #1)
   Author: J.K. Rowling
   Rating: 4.47
   Score: 0.998
   Reason:
   - Matches your favorite author
   - Rating is close to your preference

3. Harry Potter and the Philosopher's Stone (Harry Potter  #1)
   Author: J.K. Rowling
   Rating: 4.47
   Score: 0.998
   Reason:
   - Matches your favorite author
   - Rating is close to your preference

4. Carrie / 'Salem's Lot / The Shining
   Author: Stephen King
   Rating: 4.54
   Score: 0.997
   Reason:
   - Matches your favorite author
   - Rating is close to your preference

5. The Green Mile
   Author: Stephen King
   Rating: 4.44
   Score: 0.995
   Reason:
   - Matches your favorite author
   - Rating is close to your preference


In [21]:
# =========================
# UNIT TESTS (unittest Framework)
# =========================

import unittest


class TestRecommendationSystem(unittest.TestCase):

    # -------------------------
    # Test 1: Author Similarity
    # -------------------------
    def test_author_similarity(self):
        user = UserProfile()
        user.add_favorite_author("J.K. Rowling")

        book = Book("Harry Potter", "J.K. Rowling", 4.8)

        strategy = AuthorSimilarity()
        score = strategy.calculate(book, user)

        self.assertEqual(score, 1.0)


    # -------------------------
    # Test 2: Rating Similarity
    # -------------------------
    def test_rating_similarity(self):
        user = UserProfile()
        user.rate_book("Sample Book", 5)

        book = Book("Another Book", "Author", 4.5)

        strategy = RatingSimilarity()
        score = strategy.calculate(book, user)

        self.assertGreaterEqual(score, 0)
        self.assertLessEqual(score, 1)


    # -------------------------
    # Test 3: Recommendation Filtering
    # -------------------------
    def test_filter_rated_books(self):
        user = UserProfile()
        user.rate_book("Book A", 5)

        books = [
            Book("Book A", "Author1", 4.5),
            Book("Book B", "Author2", 4.0)
        ]

        controller = RecommendationController(
            [AuthorSimilarity(), RatingSimilarity()],
            [0.6, 0.4]
        )

        results = controller.recommend(books, user, top_n=5)

        # Ensure already rated book is not recommended
        titles = [rec["book"].title for rec in results]
        self.assertNotIn("Book A", titles)


    # -------------------------
    # Test 4: Recommendation Ranking
    # -------------------------
    def test_recommendation_ranking(self):
        user = UserProfile()
        user.add_favorite_author("Author1")

        books = [
            Book("Book A", "Author1", 4.5),  # should rank higher
            Book("Book B", "Author2", 4.5)
        ]

        controller = RecommendationController(
            [AuthorSimilarity()],
            [1.0]
        )

        results = controller.recommend(books, user, top_n=2)

        # First result should match preferred author
        self.assertEqual(results[0]["book"].author, "Author1")


    # -------------------------
    # Test 5: End-to-End Pipeline
    # -------------------------
    def test_full_pipeline(self):
        user = UserProfile()
        user.add_favorite_author("Author1")
        user.rate_book("Book X", 5)

        books = [
            Book("Book A", "Author1", 4.5),
            Book("Book B", "Author2", 4.0)
        ]

        controller = RecommendationController(
            [AuthorSimilarity(), RatingSimilarity()],
            [0.6, 0.4]
        )

        results = controller.recommend(books, user, top_n=2)

        self.assertTrue(len(results) > 0)
        self.assertIn("score", results[0])
        self.assertIn("explanation", results[0])


# =========================
# RUN TESTS
# =========================

if __name__ == "__main__":
    unittest.main(argv=[''], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.005s

OK
